# 02 — Exploration

## Purpose

Generate 8–12 candidate figures from `data/acled_clean.parquet`, profile the AES core dataset, and identify which findings are politically substantive enough to anchor the report. The Phase-3 deliverable here is **a menu of candidates**, not the final figures — those are produced in `03_analysis.ipynb` after we lock the report's narrative spine.

Every cell that produces a figure also prints its single-sentence headline finding so the report drafter can scan, choose, and cite. Figures are built with Plotly so they embed natively in the final HTML report.

In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from pathlib import Path

DATA = Path('../data').resolve()
FIGS = Path('../docs/figs').resolve(); FIGS.mkdir(parents=True, exist_ok=True)

aes = pd.read_parquet(DATA/'acled_clean.parquet')
coups = pd.read_parquet(DATA/'aes_coups.parquet')
ext_timeline = pd.read_parquet(DATA/'aes_external_timeline.parquet')

AES = ['Mali','Burkina Faso','Niger']
COLOURS = {'Mali':'#d62728','Burkina Faso':'#2ca02c','Niger':'#1f77b4'}

# Anchor dates (verified)
FIRST_COUP = {'Mali':'2020-08-18','Burkina Faso':'2022-01-23','Niger':'2023-07-26'}
RUSSIAN_ARRIVAL = {'Mali':'2021-12-01','Burkina Faso':'2024-01-24','Niger':'2024-04-11'}
FRENCH_EXIT = {'Mali':'2022-11-09','Burkina Faso':'2023-02-23','Niger':'2023-12-22'}

print(f'Loaded {len(aes):,} AES events; {coups.shape[0]} coup events; {ext_timeline.shape[0]} external-force markers.')

Loaded 26,977 AES events; 6 coup events; 7 external-force markers.


## 2.1  Headline numbers

In [2]:
kpi = aes.groupby('country').agg(
    events=('event_id_cnty','count'),
    fatalities=('fatalities','sum'),
    civ_targeted_events=('civilian_targeted','sum'),
    civ_fatalities=('fatalities', lambda s: aes.loc[s.index].query('civilian_targeted==True')['fatalities'].sum()),
)
kpi['civ_share_events'] = (kpi['civ_targeted_events']/kpi['events']).round(3)
kpi['civ_share_fatal']  = (kpi['civ_fatalities']/kpi['fatalities']).round(3)
print(kpi.to_string())

              events  fatalities  civ_targeted_events  civ_fatalities  civ_share_events  civ_share_fatal
country                                                                                                 
Burkina Faso   11528       29745                 3461           10054             0.300            0.338
Mali           11029       22992                 4132            9789             0.375            0.426
Niger           4420        8270                 1707            2901             0.386            0.351


## 2.2  Figure A — Monthly event count by country, with coup + Wagner-arrival markers

Headline question: *Did the post-coup, Wagner-arrival period bring a level shift in conflict events?*

In [3]:
monthly = aes.groupby(['country','year_month']).agg(events=('event_id_cnty','count'),
                                                     fatalities=('fatalities','sum')).reset_index()
fig = make_subplots(rows=3, cols=1, shared_xaxes=True, vertical_spacing=0.06,
                    subplot_titles=AES)
for i,c in enumerate(AES,1):
    sub = monthly[monthly.country==c]
    fig.add_trace(go.Scatter(x=sub.year_month, y=sub.events, mode='lines',
                              line=dict(color=COLOURS[c], width=1.6), name=c, showlegend=False), row=i, col=1)
    fig.add_vline(x=pd.Timestamp(FIRST_COUP[c]).timestamp()*1000,     line=dict(color='black', width=1.5, dash='dash'),
                  annotation_text='coup', annotation_position='top left', annotation=dict(font=dict(size=10)), row=i, col=1)
    fig.add_vline(x=pd.Timestamp(RUSSIAN_ARRIVAL[c]).timestamp()*1000, line=dict(color='#8B0000', width=1.5, dash='dot'),
                  annotation_text='Wagner / Africa Corps', annotation_position='top right', annotation=dict(font=dict(size=10, color='#8B0000')), row=i, col=1)
fig.update_layout(title='Monthly conflict events 2018–2025 (AES core), with coup and Russian-deployment dates',
                  height=620, template='plotly_white', margin=dict(t=80,r=20,b=40,l=60))
fig.update_yaxes(title_text='events / month')
fig.write_html(FIGS/'fig_A_monthly_events.html', include_plotlyjs='cdn')
fig.show()

# Headline: pre/post-coup average events per month
for c in AES:
    pre  = monthly[(monthly.country==c) & (monthly.year_month <  FIRST_COUP[c])]['events'].mean()
    post = monthly[(monthly.country==c) & (monthly.year_month >= FIRST_COUP[c])]['events'].mean()
    print(f'{c}: pre-coup {pre:.1f}/month → post-coup {post:.1f}/month (×{post/pre:.2f})')

Mali: pre-coup 75.7/month → post-coup 153.7/month (×2.03)
Burkina Faso: pre-coup 87.4/month → post-coup 185.7/month (×2.12)
Niger: pre-coup 44.3/month → post-coup 69.1/month (×1.56)


## 2.3  Figure B — Monthly civilian-targeted fatalities, by country

In [4]:
civm = aes[aes.civilian_targeted].groupby(['country','year_month'])['fatalities'].sum().reset_index()
fig = make_subplots(rows=3, cols=1, shared_xaxes=True, vertical_spacing=0.06, subplot_titles=AES)
for i,c in enumerate(AES,1):
    sub = civm[civm.country==c]
    fig.add_trace(go.Bar(x=sub.year_month, y=sub.fatalities,
                          marker_color=COLOURS[c], name=c, showlegend=False), row=i, col=1)
    fig.add_vline(x=pd.Timestamp(FIRST_COUP[c]).timestamp()*1000,     line=dict(color='black', width=1.5, dash='dash'), row=i, col=1)
    fig.add_vline(x=pd.Timestamp(RUSSIAN_ARRIVAL[c]).timestamp()*1000, line=dict(color='#8B0000', width=1.5, dash='dot'), row=i, col=1)
fig.update_layout(title='Monthly civilian-targeted fatalities (AES core)', height=620,
                  template='plotly_white', margin=dict(t=80,r=20,b=40,l=60))
fig.update_yaxes(title_text='reported fatalities')
fig.write_html(FIGS/'fig_B_civilian_fatalities_monthly.html', include_plotlyjs='cdn')
fig.show()

## 2.4  Figure C — Civilian-targeting fatality SHARE by perpetrator role, over time (12-month rolling)

This is the key descriptive figure for the research question — it visualises whether state-led civilian targeting grew (or declined) relative to non-state-led civilian targeting around the Wagner-arrival breakpoint.

In [5]:
civ = aes[aes.civilian_targeted].copy()
monthly_role = civ.groupby(['country','year_month','actor1_role'])['fatalities'].sum().unstack(fill_value=0)
monthly_role['total'] = monthly_role.sum(axis=1)
for col in ['state','non_state_armed_group','external_force']:
    if col not in monthly_role.columns:
        monthly_role[col] = 0
monthly_role = monthly_role.reset_index()

# Plot raw absolute monthly fatalities by perpetrator role, stacked.
fig = make_subplots(rows=3, cols=1, shared_xaxes=True, vertical_spacing=0.06, subplot_titles=AES)
for i,c in enumerate(AES,1):
    sub = monthly_role[monthly_role.country==c]
    for role, colour in [('state','#d62728'),('non_state_armed_group','#7f7f7f'),('external_force','#8B0000')]:
        fig.add_trace(go.Bar(x=sub.year_month, y=sub[role], name=role,
                              marker_color=colour, showlegend=(i==1)), row=i, col=1)
    fig.add_vline(x=pd.Timestamp(FIRST_COUP[c]).timestamp()*1000, line=dict(color='black', width=1.5, dash='dash'), row=i, col=1)
    fig.add_vline(x=pd.Timestamp(RUSSIAN_ARRIVAL[c]).timestamp()*1000, line=dict(color='#8B0000', width=1.5, dash='dot'), row=i, col=1)
fig.update_layout(barmode='stack', height=680, template='plotly_white',
                  title='Civilian-targeted fatalities by perpetrator role (monthly, stacked)',
                  margin=dict(t=80,r=20,b=40,l=60))
fig.update_yaxes(title_text='reported fatalities')
fig.write_html(FIGS/'fig_C_civilian_fatalities_by_role.html', include_plotlyjs='cdn')
fig.show()

## 2.5  Figure D — Pre/post Wagner comparison normalised per month (controls for window asymmetry)

Because the post-Wagner window is short (15 months for BF, 12 for Niger) but the pre-Wagner window is long (3.5–6 years), raw totals mislead. Normalising to *monthly average rates* makes the comparison fair.

In [6]:
ROW_TYPES = ['Total fatalities','Civilian-targeted fatalities','State-perpetrator civilian fatalities',
             'External-force civilian fatalities','Non-state civilian fatalities']

rows = []
for c in AES:
    sub = aes[aes.country==c]
    n_pre  = (pd.to_datetime(RUSSIAN_ARRIVAL[c]) - sub.event_date.min()).days/30.0
    n_post = (sub.event_date.max() - pd.to_datetime(RUSSIAN_ARRIVAL[c])).days/30.0
    pre  = sub[sub.event_date <  pd.to_datetime(RUSSIAN_ARRIVAL[c])]
    post = sub[sub.event_date >= pd.to_datetime(RUSSIAN_ARRIVAL[c])]
    civ_pre,  civ_post  = pre[pre.civilian_targeted], post[post.civilian_targeted]
    rows += [
        {'country':c,'metric':'Total fatalities',                       'pre/mo':pre.fatalities.sum()/n_pre,  'post/mo':post.fatalities.sum()/n_post},
        {'country':c,'metric':'Civilian-targeted fatalities',           'pre/mo':civ_pre.fatalities.sum()/n_pre,'post/mo':civ_post.fatalities.sum()/n_post},
        {'country':c,'metric':'State-perpetrator civilian fatalities',  'pre/mo':civ_pre[civ_pre.actor1_role=='state'].fatalities.sum()/n_pre,
                                                                         'post/mo':civ_post[civ_post.actor1_role=='state'].fatalities.sum()/n_post},
        {'country':c,'metric':'External-force civilian fatalities',     'pre/mo':civ_pre[civ_pre.actor1_role=='external_force'].fatalities.sum()/n_pre,
                                                                         'post/mo':civ_post[civ_post.actor1_role=='external_force'].fatalities.sum()/n_post},
        {'country':c,'metric':'Non-state civilian fatalities',           'pre/mo':civ_pre[civ_pre.actor1_role=='non_state_armed_group'].fatalities.sum()/n_pre,
                                                                         'post/mo':civ_post[civ_post.actor1_role=='non_state_armed_group'].fatalities.sum()/n_post},
    ]
rates = pd.DataFrame(rows)
rates['ratio'] = (rates['post/mo']/rates['pre/mo']).round(2)
rates['pre/mo'] = rates['pre/mo'].round(1)
rates['post/mo'] = rates['post/mo'].round(1)
print('Pre-Russian vs Post-Russian, MONTHLY-RATE comparison:')
print(rates.to_string(index=False))

Pre-Russian vs Post-Russian, MONTHLY-RATE comparison:
     country                                metric  pre/mo  post/mo  ratio
        Mali                      Total fatalities   172.1    357.7   2.08
        Mali          Civilian-targeted fatalities    66.4    160.1   2.41
        Mali State-perpetrator civilian fatalities    10.6     82.4   7.80
        Mali    External-force civilian fatalities     2.3     13.7   6.02
        Mali         Non-state civilian fatalities    53.5     63.7   1.19
Burkina Faso                      Total fatalities   274.9    620.7   2.26
Burkina Faso          Civilian-targeted fatalities    98.3    183.7   1.87
Burkina Faso State-perpetrator civilian fatalities    30.4     82.3   2.70
Burkina Faso    External-force civilian fatalities     0.0      0.2   4.84
Burkina Faso         Non-state civilian fatalities    67.5    101.1   1.50
       Niger                      Total fatalities    86.3    133.1   1.54
       Niger          Civilian-targeted fatali

### Plot the normalised pre/post comparison

Visualises the rate-comparison table as a grouped bar chart, faceted by country. Reading: any post/pre bar above 1.0 is a rate increase post-Wagner; bars below 1.0 indicate decline.

In [7]:
# Plot the normalised comparison as a grouped bar chart
fig = px.bar(rates, x='metric', y=['pre/mo','post/mo'], barmode='group',
             facet_col='country', facet_col_wrap=3,
             color_discrete_map={'pre/mo':'#1f77b4','post/mo':'#8B0000'},
             title='Pre vs Post Wagner/Africa-Corps arrival — monthly fatality rate (AES core)',
             height=520, template='plotly_white')
fig.update_xaxes(tickangle=-30)
fig.for_each_annotation(lambda a: a.update(text=a.text.split('=')[-1]))
fig.write_html(FIGS/'fig_D_pre_post_wagner_rate.html', include_plotlyjs='cdn')
fig.show()

## 2.6  Figure E — V-Dem regime trajectory with coup markers

Adds the regime-collapse context to the conflict story: Mali → closed autocracy in 2021; Burkina Faso → 2023; Niger → 2024.

In [8]:
vdem = aes[['country','year','v2x_polyarchy','v2x_libdem','v2x_regime']].drop_duplicates().sort_values(['country','year'])
fig = make_subplots(rows=1, cols=2, subplot_titles=('Polyarchy index (electoral democracy, 0–1)',
                                                    'Regime category (0=closed autocracy, 3=liberal democracy)'))
for c in AES:
    sub = vdem[vdem.country==c]
    fig.add_trace(go.Scatter(x=sub.year, y=sub.v2x_polyarchy, mode='lines+markers', name=c,
                              line=dict(color=COLOURS[c], width=2.4)), row=1, col=1)
    fig.add_trace(go.Scatter(x=sub.year, y=sub.v2x_regime, mode='lines+markers', name=c,
                              line=dict(color=COLOURS[c], width=2.4, dash='dot'), showlegend=False), row=1, col=2)
fig.update_layout(title='V-Dem v16 regime trajectory of the AES core, 2018–2025',
                  template='plotly_white', height=460)
fig.write_html(FIGS/'fig_E_vdem_regime.html', include_plotlyjs='cdn')
fig.show()

## 2.7  Figure F — Top admin1 fatality concentration (geographic hotspots)

In [9]:
admin = aes.groupby(['country','admin1'])['fatalities'].sum().reset_index().sort_values('fatalities', ascending=False)
top = admin.groupby('country').head(8)
fig = px.bar(top.sort_values(['country','fatalities']),
             x='fatalities', y='admin1', color='country',
             color_discrete_map=COLOURS, orientation='h',
             facet_col='country', facet_col_wrap=3,
             title='Top admin1 regions by reported fatalities, 2018–2025 (AES core)',
             height=520, template='plotly_white')
fig.update_yaxes(matches=None, showticklabels=True)
fig.update_xaxes(matches=None)
fig.for_each_annotation(lambda a: a.update(text=a.text.split('=')[-1]))
fig.write_html(FIGS/'fig_F_admin1_hotspots.html', include_plotlyjs='cdn')
fig.show()

## 2.8  Figure G — Geographic event scatter, coloured by post-Wagner status (Mali only)

Shows whether civilian violence shifted geographically after Wagner deployed — central Mali (Mopti, Ségou) vs north (Timbuktu, Gao).

In [10]:
mali = aes[(aes.country=='Mali') & aes.civilian_targeted].copy()
mali['phase'] = np.where(mali.post_russian_arrival, 'post-Wagner', 'pre-Wagner')
fig = px.scatter_mapbox(mali, lat='latitude', lon='longitude',
                        color='phase', size='fatalities', hover_data=['admin1','event_date','actor1','fatalities'],
                        mapbox_style='carto-positron', zoom=4.4, opacity=0.55,
                        color_discrete_map={'pre-Wagner':'#1f77b4','post-Wagner':'#8B0000'},
                        title='Mali civilian-targeted events, pre/post Wagner deployment',
                        height=600)
fig.write_html(FIGS/'fig_G_mali_geographic.html', include_plotlyjs='cdn')
fig.show()

/var/folders/pf/h0cq22td12j53pmvt_yvc2c00000gn/T/ipykernel_34281/3593020409.py:3: DeprecationWarning: *scatter_mapbox* is deprecated! Use *scatter_map* instead. Learn more at: https://plotly.com/python/mapbox-to-maplibre/
  fig = px.scatter_mapbox(mali, lat='latitude', lon='longitude',


## 2.9  Figure H — Event-type composition shift, pre vs post first coup

In [11]:
comp = aes.groupby(['country','post_first_coup','event_type'])['event_id_cnty'].count().reset_index(name='events')
comp['phase'] = np.where(comp.post_first_coup,'post-coup','pre-coup')
comp_pct = comp.groupby(['country','phase','event_type'])['events'].sum().reset_index()
comp_pct['share'] = comp_pct.groupby(['country','phase'])['events'].transform(lambda s: s/s.sum())
fig = px.bar(comp_pct, x='phase', y='share', color='event_type',
             facet_col='country',
             title='Event-type composition pre vs post first coup (share of events)',
             height=520, template='plotly_white')
fig.for_each_annotation(lambda a: a.update(text=a.text.split('=')[-1]))
fig.update_yaxes(title='share', tickformat='.0%')
fig.write_html(FIGS/'fig_H_event_composition.html', include_plotlyjs='cdn')
fig.show()

## 2.10  Quick descriptives for inline use

In [12]:
stats = {}
stats['total_events_aes']      = len(aes)
stats['total_fatalities_aes']  = int(aes.fatalities.sum())
stats['civ_events_aes']        = int(aes.civilian_targeted.sum())
stats['civ_fatalities_aes']    = int(aes[aes.civilian_targeted].fatalities.sum())
stats['date_min']              = str(aes.event_date.min().date())
stats['date_max']              = str(aes.event_date.max().date())
stats['mali_civ_pre_post']     = (int(aes[(aes.country=='Mali')&aes.civilian_targeted&~aes.post_russian_arrival].fatalities.sum()),
                                  int(aes[(aes.country=='Mali')&aes.civilian_targeted&aes.post_russian_arrival].fatalities.sum()))
stats['mali_state_pre_post']   = (int(aes[(aes.country=='Mali')&aes.civilian_targeted&~aes.post_russian_arrival&(aes.actor1_role=='state')].fatalities.sum()),
                                  int(aes[(aes.country=='Mali')&aes.civilian_targeted&aes.post_russian_arrival&(aes.actor1_role=='state')].fatalities.sum()))
stats['mali_ext_pre_post']     = (int(aes[(aes.country=='Mali')&aes.civilian_targeted&~aes.post_russian_arrival&(aes.actor1_role=='external_force')].fatalities.sum()),
                                  int(aes[(aes.country=='Mali')&aes.civilian_targeted&aes.post_russian_arrival&(aes.actor1_role=='external_force')].fatalities.sum()))
for k,v in stats.items():
    print(f'{k}: {v}')

total_events_aes: 26977
total_fatalities_aes: 61007
civ_events_aes: 9300
civ_fatalities_aes: 22744
date_min: 2018-01-01
date_max: 2025-04-25
mali_civ_pre_post: (3165, 6624)
mali_state_pre_post: (503, 3407)
mali_ext_pre_post: (108, 565)


## 2.11  Figure menu — final report selection (politics-expert flag)

**Recommended core 4–5 figures for the report (after politics-expert review):**

1. **Figure A** (monthly events with coup + Wagner vlines) — establishes the temporal scope and the regime breakpoints. *Anchor figure for the Findings section opener.*
2. **Figure C** (civilian-targeting fatalities by perpetrator role, monthly) — **the killer figure**: shows state and external-force perpetration rising sharply post-Wagner in Mali; non-state targeting roughly flat. Visualises the central mechanism.
3. **Figure D** (pre/post Wagner monthly rate) — controls for window-length asymmetry; lets the report make defensible cross-country comparisons.
4. **Figure E** (V-Dem regime trajectory) — adds the regime-collapse context; ties to the coup-contagion literature.
5. **Figure G** (Mali geographic shift) — *optional 5th* — shows whether the violence remained in central Mali or shifted geographically.

Figures A, C, D, E will go into `03_analysis.ipynb` for final polish; figure G is conditional on whether the geographic shift is striking enough to merit space.